In [1]:
import datetime as dt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import APSIMGraphHelpers as AGH
import GraphHelpers as GH
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.dates as mdates
import MathsUtilities as MUte
import shlex # package to construct the git command to subprocess format
import subprocess 
import xmltodict, json
import sqlite3
from scipy.optimize import basinhopping
from scipy.optimize import dual_annealing
from skopt import gp_minimize
%matplotlib inline

In [2]:
con = sqlite3.connect(r'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\Wheat.db')
DailyReport = pd.read_sql("Select * from DailyReport",con)

In [3]:
Simulations = pd.read_sql("Select * from _Simulations",con)
Simulations.set_index('ID',inplace=True)

In [29]:
Observed = pd.read_sql("Select * from Observed",con).dropna(axis=1,how='all')
Observed.loc[:,'SimulationName'] = [Simulations.loc[x,'Name'] for x in Observed.SimulationID]
Observed.set_index(['SimulationName','Clock.Today'],drop=False,inplace=True)
Observed.sort_index(inplace=True)
Observed.sort_index(inplace=True,axis=1)


In [13]:
Factors = pd.read_sql("Select * from _Factors",
                    con)
Factors.set_index('SimulationID',drop=False,inplace=True)
Factors = Factors.sort_values(by=['FactorName']).drop_duplicates()
Factors.sort_index(inplace=True)
Factors.sort_index(inplace=True,axis=1)

In [20]:
Simulations

,Name,FolderName
ID,,
1,APS26NRate0WaterDry,SE Queensland
2,APS26NRate40WaterDry,SE Queensland
3,APS26NRate40WaterWet,SE Queensland
4,APS26NRate0WaterWet,SE Queensland
5,APS26NRate80WaterWet,SE Queensland
...,...,...
7276,YanYean2020TOS6CvWhistler,None
7277,YanYean2020TOS6CvWills,None
7278,YanYean2020TOS6CvWyalkatchem,None


In [17]:
Factors

,CheckpointID,ExperimentName,FactorName,FactorValue,FolderName,SimulationID
SimulationID,,,,,,
1,1,APS26,Water,Dry,SE Queensland,1
1,1,APS26,NRate,0,SE Queensland,1
2,1,APS26,NRate,40,SE Queensland,2
2,1,APS26,Water,Dry,SE Queensland,2
3,1,APS26,NRate,40,SE Queensland,3
...,...,...,...,...,...,...
6499,1,YanYean2020,Cv,Axe,NPIField2020,6499
6500,1,CO2TE,CO2,350ppm,CO2AndTranspirationEfficiency,6500
6501,1,YanYean2020,TOS,7,NPIField2020,6501


In [33]:
list(Observed.loc[Observed.loc[:,'Wheat.Phenology.CurrentStageName']=='HarvestRipe',:].dropna(axis=1,how='all').columns)

['([Wheat].Leaf.Transpiration + [Soil].SoilWater.Es + [MicroClimate].PrecipitationInterception)',
 'Canopy',
 'CheckpointID',
 'Clock.Today',
 'Fungicide',
 'Grazed',
 'Mgmt',
 'N',
 'NDVIModel.Script.NDVI',
 'NDVIModel.Script.NDVI.se',
 'Nutrition',
 'PGR',
 'Potential',
 'Seeds',
 'SimulationID',
 'SimulationName',
 'Soil.Water.Volumetric(1)',
 'Soil.Water.Volumetric(2)',
 'Soil.Water.Volumetric(3)',
 'Soil.Water.Volumetric(4)',
 'Soil.Water.Volumetric(5)',
 'Soil.Water.Volumetric(6)',
 'Soil.Water.Volumetric(7)',
 'Soil.Water.Volumetric(8)',
 'Wheat.AboveGround.N',
 'Wheat.AboveGround.NError',
 'Wheat.AboveGround.Nconc',
 'Wheat.AboveGround.Nconc.se',
 'Wheat.AboveGround.Wt',
 'Wheat.AboveGround.Wt.se',
 'Wheat.AboveGround.WtError',
 'Wheat.DaysAfterSowing',
 'Wheat.Ear.N',
 'Wheat.Ear.Nconc',
 'Wheat.Ear.Wt',
 'Wheat.Grain.Density',
 'Wheat.Grain.Density.se',
 'Wheat.Grain.FWt',
 'Wheat.Grain.FWt15',
 'Wheat.Grain.Moisture',
 'Wheat.Grain.Moisture.se',
 'Wheat.Grain.N',
 'Wheat.Gra

In [45]:
for i in Observed.index:
    simulationID = Observed.loc[i,'SimulationID']
    Observed.loc[i,['ExperimentName', 'FactorName', 'FactorValue','FolderName']] = Factors.loc[simulationID,['ExperimentName', 'FactorName', 'FactorValue','FolderName']]

ValueError: Incompatible indexer with Series